In [1]:
import pandas as pd
import numpy as np

Using the Path library to read the file

In [2]:
from pathlib import Path

project_root = Path.cwd().parent.parent
data_path = project_root / "data" / "raw" / "cc_calls.csv"

df_cc = pd.read_csv(data_path, header=0)
df_cc.head(10)

,Contact_ID,Call_Date,Direction,cc_care_package,cc_care_package_discussed,cc_urgency_getting_on_site,cc_external_consultant,cc_agent_cross_sell_attempt,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,...,cc_contractor_sentiment_overall_score,cc_contractor_sentiment_issues_score,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,Co_Ref,Analysed_Call,Call_Year
0,6.255130e+11,08-05-2025,OUT_BOUND,Standard,Yes,No,No,No,Yes,Yes,...,30,20,Yes,Yes,No,Yes,Yes,HV3323,1,2025
1,5.910870e+11,25-11-2024,OUT_BOUND,Standard,Yes,No,No,No,Yes,No,...,0,0,Yes,Yes,No,Yes,Yes,PJ7066,1,2024
2,5.650910e+11,23-10-2024,IN_BOUND,Standard,Yes,No,No,No,Yes,No,...,40,20,Yes,Yes,No,Yes,Yes,DP6030,1,2024
3,5.939750e+11,13-01-2025,IN_BOUND,Premier,Yes,No,No,No,Yes,Yes,...,40,30,Yes,Yes,Yes,Yes,Yes,AM2413,1,2025
4,6.222820e+11,19-03-2025,IN_BOUND,Standard,Yes,No,No,No,Yes,Yes,...,40,20,Yes,Yes,No,Yes,Yes,ED6707,1,2025
5,6.803160e+11,11-08-2025,OUT_BOUND,Standard,Yes,No,No,No,Yes,Yes,...,40,20,Yes,Yes,No,Yes,Yes,HA1852,1,2025
6,6.799550e+11,06-08-2025,IN_BOUND,Standard,Yes,Yes,No,No,Yes,No,...,20,10,Yes,Yes,No,Yes,Yes,PS9125,1,2025
7,5.927480e+11,20-12-2024,IN_BOUND,Standard,Yes,No,No,No,Yes,Yes,...,60,60,Yes,Yes,No,Yes,Yes,YN3872,1,2024
8,6.521910e+11,17-06-2025,IN_BOUND,Premier,Yes,No,No,No,Yes,Yes,...,40,40,Yes,Yes,No,Yes,Yes,AP7473,1,2025
9,6.884250e+11,25-11-2025,OUT_BOUND,Standard,Yes,No,No,No,Yes,No,...,40,20,No,No,No,Yes,Yes,IP6445,1,2025


This dataset contains customer care call data, tracking metrics such as care package details, questionnaire completion, platform issues, contractor sentiment scores, pricing mentions, and various complaints or concerns discussed during the call.
These features will help predict whether customers are at risk of churning after their subscription due date.

In [3]:
df_cc.dtypes

Contact_ID                                  float64
Call_Date                                    object
Direction                                    object
cc_care_package                              object
cc_care_package_discussed                    object
cc_urgency_getting_on_site                   object
cc_external_consultant                       object
cc_agent_cross_sell_attempt                  object
cc_customer_issues_concerns                  object
cc_business_struggles_financial_hardship     object
cc_call_initiated_by                         object
cc_questionnaire_completion                  object
cc_chasing_response                          object
cc_issues_within_questionnaire               object
cc_login_issues                              object
cc_platform_issues                           object
cc_dissatisfaction_time_to_complete          object
cc_process_complexity_concerns               object
cc_questions_harder_than_expected            object
cc_dissatisf

In [4]:
df_cc.shape

(32882, 33)

Drop columns that are completely null or have unnamed index formats.

In [5]:
# Drop if every value in the column is missing
df_cc = df_cc.dropna(axis=1, how='all')

# Remove Unnamed columns
df_cc = df_cc.loc[:, ~df_cc.columns.str.contains('^Unnamed')]

print(f"Columns remaining: {df_cc.columns.tolist()}")

Columns remaining: ['Contact_ID', 'Call_Date', 'Direction', 'cc_care_package', 'cc_care_package_discussed', 'cc_urgency_getting_on_site', 'cc_external_consultant', 'cc_agent_cross_sell_attempt', 'cc_customer_issues_concerns', 'cc_business_struggles_financial_hardship', 'cc_call_initiated_by', 'cc_questionnaire_completion', 'cc_chasing_response', 'cc_issues_within_questionnaire', 'cc_login_issues', 'cc_platform_issues', 'cc_dissatisfaction_time_to_complete', 'cc_process_complexity_concerns', 'cc_questions_harder_than_expected', 'cc_dissatisfaction_support', 'cc_contractor_sentiment', 'cc_contractor_sentiment_start_score', 'cc_contractor_sentiment_end_score', 'cc_contractor_sentiment_overall_score', 'cc_contractor_sentiment_issues_score', 'cc_pricing_mentioned', 'cc_pricing_sentiment_impact', 'cc_refund_discussed', 'cc_contractor_suggest_leave', 'cc_contractor_complained', 'Co_Ref', 'Analysed_Call', 'Call_Year']


Check for null values for every column

In [6]:
print(df_cc.isnull().sum())

Contact_ID                                     0
Call_Date                                      0
Direction                                      0
cc_care_package                              138
cc_care_package_discussed                    138
cc_urgency_getting_on_site                   138
cc_external_consultant                       138
cc_agent_cross_sell_attempt                  138
cc_customer_issues_concerns                  138
cc_business_struggles_financial_hardship     138
cc_call_initiated_by                         138
cc_questionnaire_completion                   32
cc_chasing_response                           32
cc_issues_within_questionnaire               466
cc_login_issues                               33
cc_platform_issues                            33
cc_dissatisfaction_time_to_complete           33
cc_process_complexity_concerns                38
cc_questions_harder_than_expected             37
cc_dissatisfaction_support                    36
cc_contractor_sentim

Checking unique values for each column to understand how the data looks before cleaning

In [7]:
for col in df_cc.columns:
    if col not in ['Co_Ref', 'Contact_ID']:
        uc = df_cc[col].nunique()
        print(f"\n--- {col} ({uc} unique) ---")
        if uc < 20:
            print(df_cc[col].value_counts(dropna=False))
        else:
            print(df_cc[col].value_counts(dropna=False).head(15))


--- Call_Date (323 unique) ---
Call_Date
20-10-2025    298
05-01-2026    282
08-10-2025    248
06-01-2026    248
13-10-2025    247
10-11-2025    245
06-10-2025    239
07-10-2025    230
22-10-2025    230
26-11-2025    225
21-10-2025    220
29-10-2025    218
11-11-2025    218
27-10-2025    217
07-01-2026    216
Name: count, dtype: int64

--- Direction (2 unique) ---
Direction
OUT_BOUND    24871
IN_BOUND      8011
Name: count, dtype: int64

--- cc_care_package (57 unique) ---
cc_care_package
Not Discussed                               25879
Standard                                     3661
Express                                      2547
Premier                                       446
NaN                                           138
Assisted                                      124
[Standard/Premier/Express/Not Discussed]       23
Primary                                         6
Standards                                       5
Fast Track                                      3
Stamp

### Resolving missing Co_Ref values

There are 1196 null Co_Ref values. We check if the same Contact_ID appears in other rows that have a valid Co_Ref. If found, we use that Co_Ref. If not, we fill with a placeholder 'UNKNOWN_' + Contact_ID.

In [8]:
print(f"Null Co_Ref before: {df_cc['Co_Ref'].isnull().sum()}")

# Build a lookup from rows that have a valid Co_Ref
has_coref = df_cc[df_cc['Co_Ref'].notna()]
contact_to_coref = has_coref.drop_duplicates(subset='Contact_ID').set_index('Contact_ID')['Co_Ref']

# Fill missing Co_Ref by looking up Contact_ID
null_mask = df_cc['Co_Ref'].isna()
df_cc.loc[null_mask, 'Co_Ref'] = df_cc.loc[null_mask, 'Contact_ID'].map(contact_to_coref)

print(f"Null Co_Ref after Contact_ID lookup: {df_cc['Co_Ref'].isnull().sum()}")

Null Co_Ref before: 1196
Null Co_Ref after Contact_ID lookup: 2


In [9]:
# For remaining nulls, fill with UNKNOWN_ + Contact_ID
still_null = df_cc['Co_Ref'].isna()
df_cc.loc[still_null, 'Co_Ref'] = 'UNKNOWN_' + df_cc.loc[still_null, 'Contact_ID'].astype('Int64').astype(str)

print(f"Null Co_Ref after placeholder fill: {df_cc['Co_Ref'].isnull().sum()}")
print(f"\nPlaceholder entries:")
print(df_cc[df_cc['Co_Ref'].str.startswith('UNKNOWN_', na=False)][['Contact_ID', 'Co_Ref']])

Null Co_Ref after placeholder fill: 0

Placeholder entries:
         Contact_ID                Co_Ref
664    6.897190e+11  UNKNOWN_689719000000
31420  6.245380e+11  UNKNOWN_624538000000


### Handling NULL values

We fill nulls differently depending on the column group:
- **Care Package fields** -> fill with 'Not Discussed'
- **Sentiment fields** -> fill with 'Neutral'
- **Technical/Questionnaire issue fields** -> fill with 'No' (no issue reported)

In [10]:
# Care Package related columns - fill null with 'Not Discussed'
care_pkg_cols = ['cc_care_package', 'cc_care_package_discussed']
for col in care_pkg_cols:
    df_cc[col] = df_cc[col].fillna('Not Discussed')

# Call-related general columns - fill null with 'Not Discussed'
general_cols = [
    'cc_urgency_getting_on_site', 'cc_external_consultant',
    'cc_agent_cross_sell_attempt', 'cc_customer_issues_concerns',
    'cc_business_struggles_financial_hardship', 'cc_call_initiated_by',
    'cc_pricing_mentioned', 'cc_pricing_sentiment_impact',
    'cc_refund_discussed', 'cc_contractor_suggest_leave',
    'cc_contractor_complained'
]
for col in general_cols:
    df_cc[col] = df_cc[col].fillna('Not Discussed')

# Sentiment columns - fill null with 'Neutral'
sentiment_cols = ['cc_contractor_sentiment', 'cc_dissatisfaction_support']
for col in sentiment_cols:
    df_cc[col] = df_cc[col].fillna('Neutral')

# Sentiment score columns - fill null with median later during numeric conversion
# (handled in score cleaning step below)

# Technical / Questionnaire issue columns - fill null with 'No' (no issue)
tech_cols = [
    'cc_questionnaire_completion', 'cc_chasing_response',
    'cc_issues_within_questionnaire', 'cc_login_issues',
    'cc_platform_issues', 'cc_dissatisfaction_time_to_complete',
    'cc_process_complexity_concerns', 'cc_questions_harder_than_expected'
]
for col in tech_cols:
    df_cc[col] = df_cc[col].fillna('No')

print("Null counts after grouped fill:")
print(df_cc.isnull().sum())

Null counts after grouped fill:
Contact_ID                                   0
Call_Date                                    0
Direction                                    0
cc_care_package                              0
cc_care_package_discussed                    0
cc_urgency_getting_on_site                   0
cc_external_consultant                       0
cc_agent_cross_sell_attempt                  0
cc_customer_issues_concerns                  0
cc_business_struggles_financial_hardship     0
cc_call_initiated_by                         0
cc_questionnaire_completion                  0
cc_chasing_response                          0
cc_issues_within_questionnaire               0
cc_login_issues                              0
cc_platform_issues                           0
cc_dissatisfaction_time_to_complete          0
cc_process_complexity_concerns               0
cc_questions_harder_than_expected            0
cc_dissatisfaction_support                   0
cc_contractor_sentiment     

Cleaning Yes/No flag columns - standardizing free-text comments, stray numeric scores, and inconsistent entries into proper Yes/No/Not Discussed flags.

In [11]:
flag_cols = [
    'cc_care_package_discussed', 'cc_urgency_getting_on_site', 'cc_external_consultant',
    'cc_agent_cross_sell_attempt', 'cc_customer_issues_concerns',
    'cc_business_struggles_financial_hardship', 'cc_questionnaire_completion',
    'cc_chasing_response', 'cc_login_issues', 'cc_platform_issues',
    'cc_dissatisfaction_time_to_complete', 'cc_process_complexity_concerns',
    'cc_questions_harder_than_expected', 'cc_dissatisfaction_support',
    'cc_pricing_mentioned', 'cc_pricing_sentiment_impact',
    'cc_refund_discussed'
]

valid_flags = ['Yes', 'No', 'Not Discussed', 'Neutral']

for col in flag_cols:
    if col in df_cc.columns:
        df_cc[col] = df_cc[col].astype(str).str.strip()
        df_cc[col] = df_cc[col].replace({'0': 'No'})
        # Anything not in valid flags (stray numbers, free-text) -> 'Yes'
        invalid_mask = ~df_cc[col].isin(valid_flags)
        df_cc.loc[invalid_mask, col] = 'Yes'

Verify flag columns now only contain expected values

In [12]:
for col in flag_cols:
    if col in df_cc.columns:
        print(f"{col}: {df_cc[col].unique()}")

cc_care_package_discussed: ['Yes' 'No' 'Not Discussed']
cc_urgency_getting_on_site: ['No' 'Yes' 'Not Discussed']
cc_external_consultant: ['No' 'Yes' 'Not Discussed']
cc_agent_cross_sell_attempt: ['No' 'Yes' 'Not Discussed']
cc_customer_issues_concerns: ['Yes' 'No' 'Not Discussed']
cc_business_struggles_financial_hardship: ['Yes' 'No' 'Not Discussed']
cc_questionnaire_completion: ['No' 'Yes']
cc_chasing_response: ['No' 'Yes']
cc_login_issues: ['No' 'Yes']
cc_platform_issues: ['Yes' 'No']
cc_dissatisfaction_time_to_complete: ['Yes' 'No']
cc_process_complexity_concerns: ['Yes' 'No']
cc_questions_harder_than_expected: ['No' 'Yes']
cc_dissatisfaction_support: ['Yes' 'No' 'Neutral']
cc_pricing_mentioned: ['Yes' 'No' 'Not Discussed']
cc_pricing_sentiment_impact: ['Yes' 'No' 'Not Discussed']
cc_refund_discussed: ['No' 'Yes' 'Not Discussed']


### Standardizing cc_issues_within_questionnaire

This column has ~50 unique variants that all mean the same thing:
- **'Not applicable...'** (20+ variants like 'Not applicable as there were no issues mentioned')
- **'Not mentioned...'** (variants like 'Not mentioned in the conversation', 'Not mentioned in the transcript')
- **'Not required...'** (variants like 'Not required as Prompt 1 is No')
- **'[Yes/No]'**, **'Not enough information'**, **'N/A'**

All of these mean there were no questionnaire issues -> mapped to **'No'**.
Any remaining free-text (actual issue descriptions) -> mapped to **'Yes'**.

In [13]:
print("Before cleaning:")
print(df_cc['cc_issues_within_questionnaire'].value_counts())

Before cleaning:
cc_issues_within_questionnaire
No                                                                                           27682
Yes                                                                                           4329
Not mentioned                                                                                  398
Not applicable                                                                                 296
Not Applicable                                                                                  59
[Yes/No]                                                                                        11
Not mentioned in the conversation                                                               10
Not applicable as there were no issues mentioned                                                 9
Not applicable as no issues were mentioned                                                       8
Not applicable to this scenario                              

In [14]:
df_cc['cc_issues_within_questionnaire'] = df_cc['cc_issues_within_questionnaire'].astype(str).str.strip()

# Use pattern matching to catch ALL 'Not applicable...', 'Not mentioned...', etc. variants
lower_vals = df_cc['cc_issues_within_questionnaire'].str.lower()

no_pattern_mask = (
    lower_vals.str.startswith('not applicable') |
    lower_vals.str.startswith('not mentioned') |
    lower_vals.str.startswith('not required') |
    lower_vals.str.startswith('not enough') |
    lower_vals.str.startswith('n/a') |
    (df_cc['cc_issues_within_questionnaire'] == '[Yes/No]') |
    (df_cc['cc_issues_within_questionnaire'] == '0') |
    (df_cc['cc_issues_within_questionnaire'] == 'Not Discussed')
)

# Apply: pattern matches -> 'No'
df_cc.loc[no_pattern_mask, 'cc_issues_within_questionnaire'] = 'No'

# Anything else that is not exactly 'Yes' or 'No' is free-text issue description -> 'Yes'
valid_issue_flags = ['Yes', 'No']
invalid_mask = ~df_cc['cc_issues_within_questionnaire'].isin(valid_issue_flags)
df_cc.loc[invalid_mask, 'cc_issues_within_questionnaire'] = 'Yes'

print("\nAfter cleaning:")
print(df_cc['cc_issues_within_questionnaire'].value_counts())


After cleaning:
cc_issues_within_questionnaire
No     28528
Yes     4354
Name: count, dtype: int64


Cleaning the cc_call_initiated_by column - standardizing 'Not Relevant' to 'Not Discussed'

In [15]:
print("Before cleaning:")
print(df_cc['cc_call_initiated_by'].value_counts())

Before cleaning:
cc_call_initiated_by
Customer                         22551
Agent                             8487
Not Relevant                      1660
Not Discussed                      138
[Agent/Customer/Not Relevant]       23
No                                  22
Not Applicable                       1
Name: count, dtype: int64


In [16]:
df_cc['cc_call_initiated_by'] = df_cc['cc_call_initiated_by'].replace({
    'Not Relevant': 'Not Discussed',
    'Not Applicable': 'Not Discussed',
    'No': 'Not Discussed'
})

# Handle [Agent/Customer/Not Relevant] placeholder
df_cc['cc_call_initiated_by'] = df_cc['cc_call_initiated_by'].apply(
    lambda x: 'Not Discussed' if x not in ['Customer', 'Agent', 'Not Discussed'] else x
)

print("After cleaning:")
print(df_cc['cc_call_initiated_by'].value_counts())

After cleaning:
cc_call_initiated_by
Customer         22551
Agent             8487
Not Discussed     1844
Name: count, dtype: int64


### Standardizing cc_care_package

This column has 57 unique values but only 4 valid package tiers: **Standard**, **Premier**, **Express**, **Assisted**.

The rest are typos/misheard words from call transcripts (e.g., 'Standards', 'Stanford', 'Slander' are all 'Standard'; 'Primary', 'Premium' are 'Premier'; 'Fast Track', 'Fastest' are 'Express'; 'Assist', 'Assistive' are 'Assisted').

Using `str.contains()` pattern matching to group similar-sounding variants into the correct tier.

In [17]:
print("Before cleaning:")
print(df_cc['cc_care_package'].value_counts())

Before cleaning:
cc_care_package
Not Discussed                                            26017
Standard                                                  3661
Express                                                   2547
Premier                                                    446
Assisted                                                   124
[Standard/Premier/Express/Not Discussed]                    23
Primary                                                      6
Standards                                                    5
Fast Track                                                   3
Stamp                                                        2
Fastest package                                              2
Assisted (currently), Standard (requested)                   1
Assisted/Standard                                            1
Assistive                                                    1
Funder Package (initially), then Express (upgraded)          1
Sister Package        

In [18]:
def normalize_care_package(val):
    if pd.isna(val):
        return 'Not Discussed'
    val_clean = val.strip()
    val_lower = val_clean.lower()
    
    # Not Discussed variants
    if val_lower in ['not discussed', 'nan']:
        return 'Not Discussed'
    if val_lower.startswith('not discussed'):
        return 'Not Discussed'
    if val_lower.startswith('['):
        return 'Not Discussed'
    
    # Standard - includes typos: Standards, Stanford, Slander, Stamp, Sender, 
    # Off-Stander, Start-up, etc.
    if val_clean == 'Standard':
        return 'Standard'
    if val_lower in ['standards', 'stanford', 'slander', 'stamp', 'stamp package',
                     'sender', 'off-stander', 'start-up package', 'class',
                     'sister package', 'rest', 'address', 'sam', 'clickers package',
                     'print', 'press package', 'thunder']:
        return 'Standard'
    
    # Premier - includes Premium, Primary
    if val_clean == 'Premier':
        return 'Premier'
    if val_lower in ['premium', 'primary']:
        return 'Premier'
    
    # Express - includes Fast Track, Fastest, Faster, Fast, Excess, FAFSA, FANDA
    if val_clean == 'Express':
        return 'Express'
    if val_lower in ['fast track', 'fastest', 'faster package', 'fastest package',
                     'fast', 'excess', 'fafsa', 'fanda']:
        return 'Express'
    
    # Assisted - includes Assist, Assistive, Assistance, Assisted Package, etc.
    if val_clean == 'Assisted':
        return 'Assisted'
    if val_lower in ['assist', 'assistive', 'assistance', 'assisted package',
                     'assisted membership', 'assisted audit']:
        return 'Assisted'
    
    # Unassisted — not the same as Assisted; map to Standard
    if val_lower == 'unassisted':
        return 'Standard'
    
    # Combo values - take the first mentioned tier
    if 'standard' in val_lower and 'premier' in val_lower:
        return 'Standard'
    if 'assisted' in val_lower and 'standard' in val_lower:
        return 'Assisted'
    if 'premier' in val_lower and 'express' in val_lower:
        return 'Premier'
    if 'express' in val_lower and 'standard' in val_lower:
        return 'Express'
    if 'funder' in val_lower or 'express' in val_lower:
        return 'Express'
    
    # Catch-all for remaining sponsored/family/gold/other edge cases
    if val_lower in ['sponsored membership', 'family', 'family package', 'gold',
                     '20 working day package']:
        return 'Standard'
    
    # Anything else unrecognizable -> Not Discussed
    return 'Not Discussed'

df_cc['cc_care_package'] = df_cc['cc_care_package'].apply(normalize_care_package)

print("\nAfter cleaning:")
print(df_cc['cc_care_package'].value_counts())


After cleaning:
cc_care_package
Not Discussed    26045
Standard          3690
Express           2561
Premier            454
Assisted           132
Name: count, dtype: int64


Cleaning the cc_contractor_sentiment column - should only have Dissatisfied, Neutral, Satisfied, Not Discussed

In [19]:
print("Before cleaning:")
print(df_cc['cc_contractor_sentiment'].value_counts())

Before cleaning:
cc_contractor_sentiment
Satisfied                                                                                                                 16248
Neutral                                                                                                                   13137
Not Discussed                                                                                                              2432
Dissatisfied                                                                                                                999
 Chris                                                                                                                        4
                                                                                                                          ...  
 and the agent eventually responded with ""Can you hear me?"""                                                                1
 Holly                                                         

In [20]:
# Free-text agent comments should map to 'Neutral' since we can't determine sentiment from them
valid_sentiments = ['Dissatisfied', 'Neutral', 'Satisfied', 'Not Discussed']
df_cc['cc_contractor_sentiment'] = df_cc['cc_contractor_sentiment'].astype(str).str.strip()
df_cc['cc_contractor_sentiment'] = df_cc['cc_contractor_sentiment'].apply(
    lambda x: x if x in valid_sentiments else 'Neutral'
)

print("After cleaning:")
print(df_cc['cc_contractor_sentiment'].value_counts())

After cleaning:
cc_contractor_sentiment
Satisfied        16248
Neutral          13203
Not Discussed     2432
Dissatisfied       999
Name: count, dtype: int64


Cleaning sentiment score columns - converting to numeric. These have 'Not Discussed', text labels mixed with actual scores.

In [21]:
score_cols = [
    'cc_contractor_sentiment_start_score',
    'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score',
    'cc_contractor_sentiment_issues_score'
]

for col in score_cols:
    print(f"\n--- {col} (before) ---")
    print(df_cc[col].value_counts().head(10))


--- cc_contractor_sentiment_start_score (before) ---
cc_contractor_sentiment_start_score
50               26693
80                3759
20                 994
40                 563
60                 490
0                  103
Not Discussed      100
Satisfied           17
100                 17
Neutral             10
Name: count, dtype: int64

--- cc_contractor_sentiment_end_score (before) ---
cc_contractor_sentiment_end_score
90               13688
80               11209
50                5223
100                847
60                 617
70                 456
0                  218
95                 147
Not Discussed      131
40                 116
Name: count, dtype: int64

--- cc_contractor_sentiment_overall_score (before) ---
cc_contractor_sentiment_overall_score
90               10533
80               10071
Not Discussed     4883
85                2528
70                1344
50                1121
60                 656
100                628
40                 222
95         

In [22]:
for col in score_cols:
    # Replace 'Not Discussed' and text labels with NaN, then coerce to numeric
    df_cc[col] = df_cc[col].replace('Not Discussed', np.nan)
    df_cc[col] = pd.to_numeric(df_cc[col], errors='coerce')
    
    # Fill NaN with the median of valid scores (neutral fill)
    median_val = df_cc[col].median()
    df_cc[col] = df_cc[col].fillna(median_val)
    print(f"{col}: median={median_val}, dtype={df_cc[col].dtype}")

cc_contractor_sentiment_start_score: median=50.0, dtype=float64


cc_contractor_sentiment_end_score: median=80.0, dtype=float64
cc_contractor_sentiment_overall_score: median=80.0, dtype=float64
cc_contractor_sentiment_issues_score: median=90.0, dtype=float64


Cleaning the cc_contractor_suggest_leave column - has free-text agent notes mixed in. Standardizing to Yes/No/Not Discussed.

In [23]:
print("Before cleaning:")
print(df_cc['cc_contractor_suggest_leave'].value_counts().head(10))

Before cleaning:
cc_contractor_suggest_leave
No                                                                                                         31894
Yes                                                                                                          820
Not Discussed                                                                                                 95
The Contractor did not respond to the Agent's actions.                                                         4
The Contractor did not respond to any actions by the Agent as the conversation was incomplete.                 2
The Contractor did not respond to any actions from the Agent.                                                  2
Contractor agreed to look into the issue and send over the correct document if necessary.                      1
The contractor did not respond to any actions as the conversation is incomplete.                               1
The Contractor did not respond to the Agent's actio

In [24]:
df_cc['cc_contractor_suggest_leave'] = df_cc['cc_contractor_suggest_leave'].astype(str).str.strip()

# 'Not Applicable' -> 'No' (they didn't suggest leaving)
df_cc['cc_contractor_suggest_leave'] = df_cc['cc_contractor_suggest_leave'].replace({'Not Applicable': 'No'})

# Anything not in valid flags is free-text = the topic was discussed = 'Yes'
valid_leave_flags = ['Yes', 'No', 'Not Discussed']
invalid_mask = ~df_cc['cc_contractor_suggest_leave'].isin(valid_leave_flags)
df_cc.loc[invalid_mask, 'cc_contractor_suggest_leave'] = 'Yes'

print("After cleaning:")
print(df_cc['cc_contractor_suggest_leave'].value_counts())

After cleaning:
cc_contractor_suggest_leave
No               31894
Yes                893
Not Discussed       95
Name: count, dtype: int64


Cleaning the cc_contractor_complained column - has free-text notes and 'Not Applicable'. Standardizing to Yes/No/Not Discussed.

In [25]:
print("Before cleaning:")
print(df_cc['cc_contractor_complained'].value_counts().head(10))

Before cleaning:
cc_contractor_complained
No                                                                                                                                                                    30352
Yes                                                                                                                                                                    2366
Not Discussed                                                                                                                                                            95
Not Applicable                                                                                                                                                           56
The Contractor was appreciative of the Agent's efforts and requested an email to confirm the resolution for their client.                                                 1
The Agent investigated the customer's payment query, checked with the renewal team, and provided p

In [26]:
df_cc['cc_contractor_complained'] = df_cc['cc_contractor_complained'].astype(str).str.strip()

# 'Not Applicable' -> 'No'
df_cc['cc_contractor_complained'] = df_cc['cc_contractor_complained'].replace({'Not Applicable': 'No'})

# Anything not in valid flags is a free-text complaint description -> 'Yes'
valid_complained_flags = ['Yes', 'No', 'Not Discussed']
invalid_mask = ~df_cc['cc_contractor_complained'].isin(valid_complained_flags)
df_cc.loc[invalid_mask, 'cc_contractor_complained'] = 'Yes'

print("After cleaning:")
print(df_cc['cc_contractor_complained'].value_counts())

After cleaning:
cc_contractor_complained
No               30408
Yes               2379
Not Discussed       95
Name: count, dtype: int64


Cleaning the Direction column - checking format

In [27]:
print(df_cc['Direction'].value_counts())

Direction
OUT_BOUND    24871
IN_BOUND      8011
Name: count, dtype: int64


Converting Call_Date to datetime format

In [28]:
df_cc['Call_Date'] = pd.to_datetime(df_cc['Call_Date'], format='%d-%m-%Y', errors='coerce')
print(df_cc['Call_Date'].dtype)
print(df_cc['Call_Date'].isnull().sum(), "null dates after conversion")

datetime64[ns]
0 null dates after conversion


Final verification - checking remaining null values and dtypes after all cleaning

In [29]:
print("Remaining null values:")
print(df_cc.isnull().sum())
print(f"\nTotal rows: {len(df_cc)}")
print(f"Total columns: {len(df_cc.columns)}")

Remaining null values:
Contact_ID                                  0
Call_Date                                   0
Direction                                   0
cc_care_package                             0
cc_care_package_discussed                   0
cc_urgency_getting_on_site                  0
cc_external_consultant                      0
cc_agent_cross_sell_attempt                 0
cc_customer_issues_concerns                 0
cc_business_struggles_financial_hardship    0
cc_call_initiated_by                        0
cc_questionnaire_completion                 0
cc_chasing_response                         0
cc_issues_within_questionnaire              0
cc_login_issues                             0
cc_platform_issues                          0
cc_dissatisfaction_time_to_complete         0
cc_process_complexity_concerns              0
cc_questions_harder_than_expected           0
cc_dissatisfaction_support                  0
cc_contractor_sentiment                     0
cc_contract

In [30]:
df_cc.dtypes

Contact_ID                                         float64
Call_Date                                   datetime64[ns]
Direction                                           object
cc_care_package                                     object
cc_care_package_discussed                           object
cc_urgency_getting_on_site                          object
cc_external_consultant                              object
cc_agent_cross_sell_attempt                         object
cc_customer_issues_concerns                         object
cc_business_struggles_financial_hardship            object
cc_call_initiated_by                                object
cc_questionnaire_completion                         object
cc_chasing_response                                 object
cc_issues_within_questionnaire                      object
cc_login_issues                                     object
cc_platform_issues                                  object
cc_dissatisfaction_time_to_complete                 obje

Giving type Category to categorical columns 

In [31]:
cat_cols = [
    'Direction', 'cc_care_package', 'cc_care_package_discussed',
    'cc_urgency_getting_on_site', 'cc_external_consultant',
    'cc_agent_cross_sell_attempt', 'cc_customer_issues_concerns',
    'cc_business_struggles_financial_hardship', 'cc_call_initiated_by',
    'cc_questionnaire_completion', 'cc_chasing_response',
    'cc_issues_within_questionnaire', 'cc_login_issues',
    'cc_platform_issues', 'cc_dissatisfaction_time_to_complete',
    'cc_process_complexity_concerns', 'cc_questions_harder_than_expected',
    'cc_dissatisfaction_support', 'cc_contractor_sentiment',
    'cc_pricing_mentioned', 'cc_pricing_sentiment_impact',
    'cc_refund_discussed', 'cc_contractor_suggest_leave',
    'cc_contractor_complained'
]

for col in cat_cols:
    df_cc[col] = df_cc[col].astype('category')

In [32]:
df_cc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32882 entries, 0 to 32881
Data columns (total 33 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   Contact_ID                                32882 non-null  float64       
 1   Call_Date                                 32882 non-null  datetime64[ns]
 2   Direction                                 32882 non-null  category      
 3   cc_care_package                           32882 non-null  category      
 4   cc_care_package_discussed                 32882 non-null  category      
 5   cc_urgency_getting_on_site                32882 non-null  category      
 6   cc_external_consultant                    32882 non-null  category      
 7   cc_agent_cross_sell_attempt               32882 non-null  category      
 8   cc_customer_issues_concerns               32882 non-null  category      
 9   cc_business_struggles_financ

In [33]:
df_cc.head()

,Contact_ID,Call_Date,Direction,cc_care_package,cc_care_package_discussed,cc_urgency_getting_on_site,cc_external_consultant,cc_agent_cross_sell_attempt,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,...,cc_contractor_sentiment_overall_score,cc_contractor_sentiment_issues_score,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,Co_Ref,Analysed_Call,Call_Year
0,6.255130e+11,2025-05-08,OUT_BOUND,Standard,Yes,No,No,No,Yes,Yes,...,30.0,20.0,Yes,Yes,No,Yes,Yes,HV3323,1,2025
1,5.910870e+11,2024-11-25,OUT_BOUND,Standard,Yes,No,No,No,Yes,No,...,0.0,0.0,Yes,Yes,No,Yes,Yes,PJ7066,1,2024
2,5.650910e+11,2024-10-23,IN_BOUND,Standard,Yes,No,No,No,Yes,No,...,40.0,20.0,Yes,Yes,No,Yes,Yes,DP6030,1,2024
3,5.939750e+11,2025-01-13,IN_BOUND,Premier,Yes,No,No,No,Yes,Yes,...,40.0,30.0,Yes,Yes,Yes,Yes,Yes,AM2413,1,2025
4,6.222820e+11,2025-03-19,IN_BOUND,Standard,Yes,No,No,No,Yes,Yes,...,40.0,20.0,Yes,Yes,No,Yes,Yes,ED6707,1,2025


Saving the cleaned dataset

In [34]:
import os

# Ensure the clean data directory exists
clean_data_path = project_root / "data" / "processed"
os.makedirs(clean_data_path, exist_ok=True)

# Write out to clean folder
out_file = clean_data_path / "cleaned_cc_calls.csv"
df_cc.to_csv(out_file, index=False)
print(f"Cleaned dataset saved successfully to {out_file}")

Cleaned dataset saved successfully to c:\Users\madha\OneDrive\Desktop\DS\Churn-Prediction-JMAN\data\processed\cleaned_cc_calls.csv
